# KG1 V82 HUIKANG RECIPE

Builds on V80 MEGA (dgxchen 0.85 replica) + V81 canonicalization, and layers the full huikang recipe (trick #1 -> +0.20 LB + tricks #2-10).

## New tricks vs V81

1. **9-module LoRA targets** inc. `lm_head` (`scripts/kg1_huikang_conversion.py`)
2. **Tinker -> Kaggle SVD merge** (gate_proj + x_proj -> in_proj @ rank 32)
3. **Expert unfusing** (MoE `experts.w1`/`w2` -> per-expert `up_proj`/`down_proj`)
4. **Key rename** `model.model` -> `model.backbone`
5. **Solver CoTs per category** (`scripts/kg1_solver_cots.py`)
6. **Rule-based CoT verification** (`scripts/kg1_verify_cots.py`)
7. **Min-logprob priority duplication** (`scripts/kg1_min_logprob_duplication.py`)
8. **Token budget <= 7600** (leaves headroom for `</think>\boxed{}` trailer)
9. **V81 canonicalization kept** (handles the 94 equation-row brace bug + 32 binary collisions)

## Expected delta

V80 base 0.84-0.85 -> V82 target **0.87-0.92** (see `docs/V82_TRICKS_ROADMAP.md`).

## Credentials (Colab secrets)

Add these in the Colab Secrets panel (lock icon):
- `HF_KEY` = Hugging Face token
- `KAGGLE_USERNAME` = felipe1983
- `KAGGLE_KEY` = from kaggle.json

Author: FELIPEACASTRO
Date: 2026-04-22

## Cell 0 - Colab secrets + workspace

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import userdata  # type: ignore
    os.environ['HF_TOKEN'] = userdata.get('HF_KEY')
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass

WORKSPACE = Path('/content/kg1-v82')
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
print('workspace:', WORKSPACE)

## Cell 1 - Clone repo

In [ ]:
REPO_DIR = WORKSPACE / 'KG1'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'claude/competent-shamir',
                    'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git', str(REPO_DIR)],
                   check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('repo:', REPO_DIR)

## Cell 2 - Install deps (V80 MEGA wheel set + safetensors)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch==2.5.1', 'transformers==4.46.0', 'peft==0.13.2',
                'accelerate==1.0.1', 'datasets==3.0.2', 'bitsandbytes==0.44.1',
                'trl==0.11.4', 'safetensors', 'scikit-learn==1.5.2',
                'kaggle==1.6.17', 'huggingface_hub==0.26.2'], check=True)

## Cell 3 - Build solver CoTs (per-category deterministic)

Replaces the raw Gemini-distilled CoTs with deterministic solver traces per family (huikang trick #6 + Donald Galliano playbook). Expected lift: bit_manipulation 22% -> 85%, cipher 100%, numeral 100%, gravity 100%, unit 100%.

In [ ]:
TRAIN_CSV = REPO_DIR / 'data' / 'train.csv'
SOLVER_JSONL = REPO_DIR / 'runs' / 'v82_solver_cots.jsonl'
subprocess.run([sys.executable, 'scripts/kg1_solver_cots.py',
                '--input', str(TRAIN_CSV),
                '--output', str(SOLVER_JSONL)], check=True)
print('solver CoTs:', SOLVER_JSONL)

## Cell 4 - Verify CoTs (drop rows whose CoT fails rule-based check)

Huikang trick #4 (+0.06 LB). Every CoT is extracted + verify()ed with per-category rules:
- bit: exact 8-bit match
- cipher / numeral: case-insensitive exact
- gravity / unit: rel_tol 0.5% + abs_tol 0.05
- equation: exact-string

In [ ]:
VERIFIED_JSONL = REPO_DIR / 'runs' / 'v82_solver_cots_verified.jsonl'
VERIFY_REPORT = REPO_DIR / 'runs' / 'v82_verify_report.json'
subprocess.run([sys.executable, 'scripts/kg1_verify_cots.py',
                '--input', str(SOLVER_JSONL),
                '--output', str(VERIFIED_JSONL),
                '--report-json', str(VERIFY_REPORT)], check=True)
import json as _json
print(_json.dumps(_json.loads(VERIFY_REPORT.read_text()), indent=2))

## Cell 5 - Canonicalize labels (V81 carry-over)

Applies the final format guard (single `\boxed{}`, equation-family `Final answer is:` prefix, strip LaTeX wrappers, strip units, enforce 8-bit zero-pad).

In [ ]:
import json
from scripts.kg1_canonicalize_output import canonicalize_answer, detect_family

CANONICAL_JSONL = REPO_DIR / 'runs' / 'v82_canonical.jsonl'
count = 0
with VERIFIED_JSONL.open('r', encoding='utf-8') as fin, \
        CANONICAL_JSONL.open('w', encoding='utf-8') as fout:
    for line in fin:
        row = json.loads(line)
        prompt = row.get('prompt', '')
        fam = detect_family(prompt)
        completion = row.get('completion', '')
        row['completion'] = canonicalize_answer(completion, family_hint=fam)
        fout.write(json.dumps(row, ensure_ascii=False) + '\n')
        count += 1
print(f'Canonicalized {count} rows -> {CANONICAL_JSONL}')

## Cell 6 - Token budget guard (<= 7600 per completion)

Trick #5: CoTs > 7600 tokens truncate the terminal `\boxed{}` and the sample silently scores 0. Drop over-length rows.

In [ ]:
from transformers import AutoTokenizer
TOK = AutoTokenizer.from_pretrained('nvidia/Nemotron-3-Nano-30B-A3B-BF16', trust_remote_code=False)

BUDGETED_JSONL = REPO_DIR / 'runs' / 'v82_budgeted.jsonl'
MAX_TOK = 7600
kept, dropped = 0, 0
with CANONICAL_JSONL.open('r', encoding='utf-8') as fin, \
        BUDGETED_JSONL.open('w', encoding='utf-8') as fout:
    for line in fin:
        row = json.loads(line)
        n = len(TOK.encode(row.get('completion', '')))
        if n <= MAX_TOK:
            fout.write(json.dumps(row, ensure_ascii=False) + '\n')
            kept += 1
        else:
            dropped += 1
print(f'Token budget: kept={kept} dropped={dropped}')

## Cell 7 - LoRA training with 9 targets incl. lm_head

Huikang trick #1: 9 target modules (`k_proj, o_proj, in_proj, q_proj, up_proj, v_proj, down_proj, out_proj, lm_head`), r=32 alpha=32 dropout=0.

Trick #11 (dgxchen 0.85 hparams):
- `per_device_train_batch_size=1` (microbatch=2 drops 0.1 LB)
- `gradient_accumulation_steps=32`
- `learning_rate=2e-4`, `lr_scheduler_type='linear'`
- `max_grad_norm=1e9` (effectively no clipping)
- `bf16=True`, `gradient_checkpointing=True` with `use_reentrant=False`
- `packing=False`, `completion_only_loss=True`

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

BASE = 'nvidia/Nemotron-3-Nano-30B-A3B-BF16'
TARGETS = ['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj',
           'down_proj', 'out_proj', 'lm_head']

model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, attn_implementation='eager',
    trust_remote_code=False)
lora_cfg = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.0,
                      target_modules=TARGETS, task_type='CAUSAL_LM')
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

ds = load_dataset('json', data_files=str(BUDGETED_JSONL), split='train')

ADAPTER_DIR = REPO_DIR / 'runs' / 'v82_adapter'
cfg = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    lr_scheduler_type='linear',
    warmup_steps=0,
    max_length=8192,
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    packing=False,
    completion_only_loss=True,
    dataloader_num_workers=0,
    logging_steps=10,
    save_steps=0,
)
trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))
print('saved:', ADAPTER_DIR)

## Cell 8 - Min-logprob priority duplication (optional second pass)

Trick #10 (+0.06 LB). Score the SFT dataset with the v82 adapter, collect ids whose `min(prompt_logprobs) < -0.69`, and duplicate them 2x for a finetune-on-finetune pass.

In [ ]:
PRIORITY = REPO_DIR / 'runs' / 'v82_priority.txt'
DUPLICATED = REPO_DIR / 'runs' / 'v82_duplicated.jsonl'

# Requires vllm - skip if not available.
try:
    import vllm  # noqa: F401
    subprocess.run([sys.executable, 'scripts/kg1_min_logprob_duplication.py', 'score',
                    '--dataset', str(BUDGETED_JSONL),
                    '--adapter', str(ADAPTER_DIR),
                    '--base-model', BASE,
                    '--priority-out', str(PRIORITY)], check=True)
    subprocess.run([sys.executable, 'scripts/kg1_min_logprob_duplication.py', 'dup',
                    '--dataset', str(BUDGETED_JSONL),
                    '--priority', str(PRIORITY),
                    '--output', str(DUPLICATED)], check=True)
    print('Priority duplication done ->', DUPLICATED)
except ImportError:
    print('vllm unavailable; skip priority duplication (safe fallback).')

## Cell 9 - Tinker -> Kaggle adapter conversion

Only runs if you trained on Tinker. For native Colab SFT, the PEFT adapter is already in Kaggle format and you can skip this cell.

Trick #1 (+0.20 LB): 9-module target list, SVD rank-32 merge, expert unfusing, key rename.

In [ ]:
TINKER_DIR = REPO_DIR / 'runs' / 'tinker_adapter'
KAGGLE_ADAPTER = REPO_DIR / 'runs' / 'v82_kaggle_adapter'
if TINKER_DIR.exists():
    subprocess.run([sys.executable, 'scripts/kg1_huikang_conversion.py',
                    '--tinker-dir', str(TINKER_DIR),
                    '--out-dir', str(KAGGLE_ADAPTER),
                    '--base-model', BASE,
                    '--rank', '32'], check=True)
    print('Tinker -> Kaggle conversion done:', KAGGLE_ADAPTER)
else:
    KAGGLE_ADAPTER = ADAPTER_DIR
    print('No Tinker run; using native PEFT adapter at', KAGGLE_ADAPTER)

## Cell 10 - Pre-score before submit (99% rule)

Trick: never submit unless predicted score > last LB + 0.005. Uses the stratified-RF pre-scorer if available.

In [ ]:
try:
    from scripts.kg1_prescore_rf import prescore_submission
    report = prescore_submission(str(KAGGLE_ADAPTER), val_subset_size=100)
    print(json.dumps({k: v for k, v in report.items() if k != 'details'}, indent=2))
    LAST_LB = 0.85  # V80 MEGA
    MIN_DELTA = 0.005
    if report['predicted_kaggle_score'] < LAST_LB + MIN_DELTA:
        raise RuntimeError(
            f"Predicted {report['predicted_kaggle_score']:.3f} < last LB {LAST_LB} + {MIN_DELTA}. Abort.")
    print('Predicted improvement confirmed - safe to submit.')
except ImportError:
    print('prescore_rf unavailable; run manual eval before submit.')

## Cell 11 - Build submission ZIP (2 files at root)

Trick #15: files at root, no nested folder. `zip -m submission.zip *`.

In [ ]:
import shutil, zipfile
SUBMISSION = REPO_DIR / 'submission.zip'
if SUBMISSION.exists():
    SUBMISSION.unlink()
with zipfile.ZipFile(str(SUBMISSION), 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ('adapter_config.json', 'adapter_model.safetensors'):
        src = KAGGLE_ADAPTER / name
        if not src.exists():
            raise FileNotFoundError(src)
        zf.write(src, arcname=name)
print('Submission:', SUBMISSION, SUBMISSION.stat().st_size, 'bytes')

## Cell 12 - Submit

Uncomment to submit to Kaggle. Respect the 5-submits-per-day cap.

In [ ]:
# subprocess.run(['kaggle', 'competitions', 'submit',
#                 '-c', 'nvidia-nemotron-model-reasoning-challenge',
#                 '-f', str(SUBMISSION),
#                 '-m', 'V82 huikang recipe + triple-check tricks'], check=True)
print('Ready to submit V82.')